In [1]:
from dataclasses import dataclass, asdict, field
from pathlib import Path
from loguru import logger
import numpy as np
import pandas as pd
import psychrolib

import combined_cooler
from solhycool_visualization.sampling import plot_samples
from phd_visualizations import save_figure

# auto reload modules
%load_ext autoreload
%autoreload 2

psychrolib.SetUnitSystem(psychrolib.SI)
get_Twb = np.vectorize(psychrolib.GetTWetBulbFromRelHum) # Vectorize the function
get_HR = np.vectorize(psychrolib.GetRelHumFromTWetBulb) # Vectorize the function


cc_model = combined_cooler.initialize()

case_studies_dict = {
    "pilot_plant_200kW": {
        "system_id": "pilot_plant",
        "n_dc": 1,
        "n_wct": 1,
        # "qwct": (6., 24.),
        "n_qwct": 10,
        "n_mw_ma_ratio": 10,
        "n_deltaTwb_Tin": 15,
    },
    "andasol_90MW": {
        "system_id": "andasol", # For fan speed to air flow correlation
        "n_mw_ma_ratio": 5,
        "mw_ma_ratio": (0.5, 4),
        "n_deltaTwb_Tin": 5,
        "n_qwct": 12,
        "wwct": (20, 100),
        "qwct": (1152, 3960), # Operating range of a single tower (there should be 3 towers)
    }
}

case_study_id: str = "andasol_90MW"
base_output_path: Path = Path("../results/model_inputs_sampling") / case_study_id
base_output_path.mkdir(parents=True, exist_ok=True)

params = case_studies_dict[case_study_id]


Authorization required, but no authorization protocol specified



# DC

In [12]:
# New approach, directly generate the samples given ranges and a resolution for the inputs

@dataclass
class ModelInputsRange:
    Tamb: tuple[float, float] = (3., 50.)
    deltaTamb_Tin: tuple[float, float] = (3., 30.)
    Tdc_in: tuple[float, float] = (25., 45.)
    qdc: tuple[float, float] = field(default_factory=lambda: (6.*params["n_dc"], 24.*params["n_dc"]))
    wdc: tuple[float, float] = (11., 99.18)

@dataclass
class NumInputUpdates:
    Tamb: int = 7
    deltaTamb_Tin: int = 7
    qdc: int = 7
    wdc: int = 6

@dataclass
class ModelInputs:
    Tamb: np.ndarray[float]
    Tdc_in: np.ndarray[float]
    qdc: np.ndarray[float]
    wdc: np.ndarray[float]

    @classmethod
    def initialize(cls, num_input_updates: NumInputUpdates = NumInputUpdates(), inputs_range: ModelInputsRange = ModelInputsRange()) -> "ModelInputs":
        # Generate base temperature combinations
        Tamb = np.linspace(inputs_range.Tamb[0], inputs_range.Tamb[1]-inputs_range.deltaTamb_Tin[0], num_input_updates.Tamb)
        deltaTamb_Tin = np.linspace(inputs_range.deltaTamb_Tin[0], inputs_range.deltaTamb_Tin[1], num_input_updates.deltaTamb_Tin)

        Tamb_grid, deltaT_grid = np.meshgrid(Tamb, deltaTamb_Tin, indexing='ij')
        Tdc_in_grid = Tamb_grid + deltaT_grid

        # Mask out invalid Tdc_in
        mask = (Tdc_in_grid <= inputs_range.Tdc_in[1]) & (Tdc_in_grid >= inputs_range.Tdc_in[0])
        Tamb_valid = Tamb_grid[mask]
        Tdc_in_valid = Tdc_in_grid[mask]

        # Generate qdc and wdc ranges
        qdc_vals = np.linspace(inputs_range.qdc[0], inputs_range.qdc[1], num_input_updates.qdc)
        wdc_vals = np.linspace(inputs_range.wdc[0], inputs_range.wdc[1], num_input_updates.wdc)

        # Create full meshgrid over all valid (Tamb, Tdc_in) × (qdc, wdc)
        Tamb_final, qdc_final, wdc_final = np.meshgrid(Tamb_valid, qdc_vals, wdc_vals, indexing='ij')
        Tdc_in_final, _, _ = np.meshgrid(Tdc_in_valid, qdc_vals, wdc_vals, indexing='ij')

        # Flatten everything
        return cls(
            Tamb = Tamb_final.ravel(),
            Tdc_in = Tdc_in_final.ravel(),
            qdc = qdc_final.ravel(),
            wdc = wdc_final.ravel()
        )

# 1. Define problem
output_path = base_output_path / "dc_in.csv"
df = pd.DataFrame(asdict(ModelInputs.initialize()))

display(df)

df.to_csv(output_path)
logger.info(f"{len(df)} samples generated and saved at {output_path}")


,Tamb,Tdc_in,qdc,wdc
0,3.000000,28.500000,2076.0,11.000
1,3.000000,28.500000,2076.0,28.636
2,3.000000,28.500000,2076.0,46.272
3,3.000000,28.500000,2076.0,63.908
4,3.000000,28.500000,2076.0,81.544
...,...,...,...,...
793,39.666667,42.666667,8304.0,28.636
794,39.666667,42.666667,8304.0,46.272
795,39.666667,42.666667,8304.0,63.908
796,39.666667,42.666667,8304.0,81.544


2025-09-08 17:24:52.390 | INFO     | __main__:<module>:62 - 798 samples generated and saved at ../results/model_inputs_sampling/andasol_75_90MW/dc_in.csv


In [13]:
rename_dict = {
    "Tamb": "T<sub>amb</sub>",
    "Tdc_in": "T<sub>dc,in</sub>",
    "qdc": "q<sub>dc</sub>",
    "wdc": "w<sub>dc</sub>"
}
df.rename(columns=rename_dict, inplace=True)

fig = plot_samples(df, Ncols=len(df.columns))
fig.update_layout(width=800)

save_figure(fig, figure_path=base_output_path, figure_name="viz_samples_dc", formats=["png", "html"], scale=3)

fig


2025-09-08 17:24:55.792 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../results/model_inputs_sampling/andasol_75_90MW/viz_samples_dc.png
2025-09-08 17:24:55.795 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../results/model_inputs_sampling/andasol_75_90MW/viz_samples_dc.html


# WCT

In [32]:
# New approach, directly generate the samples given ranges and a resolution for the inputs

def get_temperature_and_humidity_values(Twb: float, Tamb_max: float = 50, n_samples_max: int = 5, deltaTmin: float = 1.0 ) -> tuple[np.ndarray, np.ndarray]:
    """ 
    Generate temperature and humidity values based on the wet bulb temperature (Twb) and maximum ambient temperature (Tamb_max).
    The function calculates the ambient temperature (Tamb) and relative humidity (HR) values based on the given Twb.
    The function returns the valid Tamb and HR values that are within the specified range.
    """
    if Twb > Tamb_max:
        raise ValueError(f"Tamb_max ({Tamb_max:.2f})> Tw ({Twb:.2f}) °C")
    n_samples = min(int((Tamb_max - Twb) / deltaTmin), n_samples_max)
    
    Tamb = np.linspace(Twb, Tamb_max, n_samples)
    HR = np.minimum(get_HR(Tamb, np.full(n_samples,Twb), np.full(n_samples, 101325)), 0.95) 
    # Filter out invalid values
    Twb_ = get_Twb(Tamb, HR, np.full(len(Tamb), 101325))
    
    return Tamb[np.abs(Twb_ - Twb) < 1e-1], HR[np.abs(Twb_ - Twb) < 1e-1]

@dataclass
class ModelInputsRange:
    Twb: tuple[float, float] = (3., 40.)
    deltaTwb_Tin: tuple[float, float] = (3., 30.)
    Twct_in: tuple[float, float] = (25., 45.)
    qwct: tuple[float, float] = (6, 24)
    wwct: tuple[float, float] = (21.0, 93.4161)
    mw_ma_ratio: tuple[float, float] = (0.5, 4.)

@dataclass
class NumInputUpdates:
    Twb: int = 10
    deltaTwb_Tin: int = 10
    qwct: int = 10
    mw_ma_ratio: int = 10

@dataclass
class ModelInputs:
    Tamb: np.ndarray[float]
    HR: np.ndarray[float]
    Twct_in: np.ndarray[float]
    qwct: np.ndarray[float]
    wwct: np.ndarray[float]

    @classmethod
    def initialize(cls, num_input_updates: NumInputUpdates = NumInputUpdates(), inputs_range: ModelInputsRange = ModelInputsRange()) -> "ModelInputs":
        # Generate base temperature combinations
        Twb_grid, deltaT_grid = np.meshgrid(
            np.linspace(inputs_range.Twb[0], inputs_range.Twb[1], num_input_updates.Twb), 
            np.linspace(inputs_range.deltaTwb_Tin[0], inputs_range.deltaTwb_Tin[1], num_input_updates.deltaTwb_Tin), 
            indexing='ij'
        )
        Twct_in_grid = Twb_grid + deltaT_grid
        # Mask out invalid Twct_in
        mask = (Twct_in_grid <= inputs_range.Twct_in[1]) & (Twct_in_grid >= inputs_range.Twct_in[0])
        Twb_valid = Twb_grid[mask]
        Twct_in_valid = Twct_in_grid[mask]
        # Decompose Twb into Tamb and HR
        Twct_in_flat = []
        Tamb_flat = []
        HR_flat = []
        for Twb, Twct_in in zip(Twb_valid.ravel(), Twct_in_valid.ravel()):
            Tamb, HR = get_temperature_and_humidity_values(Twb)
            Tamb_flat.extend(Tamb)
            HR_flat.extend(HR*100)
            Twct_in_flat.extend( [Twct_in]*len(Tamb) )

        # # Generate base qwct and mw_ma_ratio combinations
        qwct_grid, mw_ma_ratio_grid = np.meshgrid(
            np.linspace(inputs_range.qwct[0], inputs_range.qwct[1], num_input_updates.qwct), 
            np.linspace(inputs_range.mw_ma_ratio[0], inputs_range.mw_ma_ratio[1], num_input_updates.mw_ma_ratio), 
            indexing='ij'
        )

        qwct_flat = qwct_grid.ravel()
        mw_ma_ratio_flat = mw_ma_ratio_grid.ravel()
        
        wwct, valid = zip(*[
            cc_model.air_mass_flow_rate_to_fan_speed( qwct/3.6 / mw_ma_ratio, params["system_id"], nargout=2 ) # 1/mw/ma [kg/s / kg/s] * mw [kg/s: m3/h -> 1000 kg/m³ * 1/3600 h/s] 
            # (np.random.rand(1) * 90, True)
            for qwct, mw_ma_ratio in zip(qwct_flat, mw_ma_ratio_flat)
        ])
        
        mask = np.array(valid) # np.array(valid).reshape(qwct_grid.shape) 
        qwct_valid = np.array(qwct_flat)[mask]
        wwct_valid = np.array(wwct)[mask]

        # Create full meshgrid over all valid (Tamb, HR, Twct_in) × (qwct, wwct)
        Tamb_final, qwct_final = np.meshgrid(Tamb_flat, qwct_valid, indexing='ij')
        _, wwct_final = np.meshgrid(Tamb_flat, wwct_valid, indexing='ij')
        HR_final, _ = np.meshgrid(HR_flat, qwct_valid, indexing='ij')
        Twct_in_final, _ = np.meshgrid(Twct_in_flat, qwct_valid, indexing='ij')

        # Flatten everything
        return cls(
            Tamb = Tamb_final.ravel(),
            HR = HR_final.ravel(),
            Twct_in = Twct_in_final.ravel(),
            qwct = qwct_final.ravel(),
            wwct = wwct_final.ravel()
        )

# 1. Define problem
output_path = base_output_path / "wct_in.csv"

model_inputs = ModelInputs.initialize(
    inputs_range = ModelInputsRange(
        qwct=params.get("qwct", ModelInputsRange().qwct), 
        mw_ma_ratio=params.get("mw_ma_ratio", ModelInputsRange.mw_ma_ratio),
        # wwct=(0, 100)
    ),
    num_input_updates=NumInputUpdates(
        mw_ma_ratio=params.get("n_mw_ma_ratio", NumInputUpdates.mw_ma_ratio),
        qwct=params.get("n_qwct", NumInputUpdates.qwct),
    )
)

# From model inputs, compute Twb, mw/ma
ma_kgs = np.array(cc_model.fan_speed_to_air_mass_flow_rate_fit(model_inputs.wwct, params["system_id"])[0])

additional_inputs_dict = {
    "Twb": np.array(
        get_Twb(model_inputs.Tamb, model_inputs.HR/100, np.full(len(model_inputs.Tamb), 101325))
    ),
    "mw_ma_ratio": np.array([model_inputs.qwct / (ma_kgs * 3.6)]).reshape(-1)
}

output_dict = {
    **asdict(model_inputs),
    **additional_inputs_dict,
}
df = pd.DataFrame(output_dict)

display(df)

df.to_csv(output_path)
logger.info(f"{len(df)} samples generated and saved at {output_path}")

rename_dict = {
    "Tamb": "T<sub>amb</sub>",
    "Twct_in": "T<sub>wct,in</sub>",
    "qwct": "q<sub>wct</sub>",
    "wwct": "ω<sub>wct</sub>",
    "Twb": "T<sub>wb</sub>",
    "mw_ma_ratio": "ṁ<sub>w</sub>/ṁ<sub>ma</sub>"
}
df.rename(columns=rename_dict, inplace=True)

fig = plot_samples(df, Ncols=len(df.columns))
fig.update_layout(width=1000)

fig.show()


,Tamb,HR,Twct_in,qwct,wwct,Twb,mw_ma_ratio
0,14.75,0.000971,27.0,1152.000000,32.103815,3.070432,0.500
1,14.75,0.000971,27.0,1152.000000,23.864511,3.070432,1.375
2,14.75,0.000971,27.0,1152.000000,22.151955,3.070432,2.250
3,14.75,0.000971,27.0,1152.000000,21.410259,3.070432,3.125
4,14.75,0.000971,27.0,1407.272727,35.218541,3.070432,0.500
...,...,...,...,...,...,...,...
8609,50.00,54.625129,43.0,3960.000000,92.136147,39.999675,0.500
8610,50.00,54.625129,43.0,3960.000000,35.628772,39.999675,1.375
8611,50.00,54.625129,43.0,3960.000000,28.935151,39.999675,2.250
8612,50.00,54.625129,43.0,3960.000000,26.182107,39.999675,3.125


2025-09-17 14:29:22.675 | INFO     | __main__:<module>:136 - 8614 samples generated and saved at ../results/model_inputs_sampling/andasol_90MW/wct_in.csv


In [ ]:
save_figure(fig, figure_path=base_output_path, figure_name="viz_samples_wct", formats=["png", "html"], scale=2)
logger.info(f"Figure saved at {base_output_path}")


## Evaluation

In [3]:
cc_model.wct_model_physical_andasol(
    30., 50., 40., 1800., 50., nargout=3
)


> In wct_model_physical_andasol>solve_wct_with_timeout (line 822)
In wct_model_physical_andasol (line 183)


(21.527878706118496, 34.85080000000001, 49108.689301588536)

In [2]:
from solhycool_modeling.evaluation import WCTModelEvaluator

# Configuration
timeout = 3.0  # seconds
n_processes = None  # Auto-detect

# Create evaluator
evaluator = WCTModelEvaluator(
    case_study_id=case_study_id,
    timeout=timeout,
    n_processes=n_processes,
    base_path=base_output_path
)


2025-09-18 08:02:00.093 | INFO     | solhycool_modeling.evaluation:__post_init__:110 - Successfully initialized combined_cooler model
2025-09-18 08:02:00.094 | INFO     | solhycool_modeling.evaluation:__post_init__:122 - Initialized WCT Model Evaluator:
2025-09-18 08:02:00.094 | INFO     | solhycool_modeling.evaluation:__post_init__:123 -   Case study: andasol_90MW
2025-09-18 08:02:00.095 | INFO     | solhycool_modeling.evaluation:__post_init__:124 -   Model type: andasol
2025-09-18 08:02:00.095 | INFO     | solhycool_modeling.evaluation:__post_init__:125 -   Timeout: 3.0s
2025-09-18 08:02:00.096 | INFO     | solhycool_modeling.evaluation:__post_init__:126 -   Parallel processes: 23
2025-09-18 08:02:00.097 | INFO     | solhycool_modeling.evaluation:__post_init__:127 -   MATLAB model initialized: True
2025-09-18 08:02:00.094 | INFO     | solhycool_modeling.evaluation:__post_init__:122 - Initialized WCT Model Evaluator:
2025-09-18 08:02:00.094 | INFO     | solhycool_modeling.evaluation:_

In [ ]:
# Run evaluation
evaluator.run_evaluation(use_existing_input=True)


### Process / visualize results

In [7]:
wct_out_raw = pd.read_csv(base_output_path / "wct_out_raw.csv", index_col=0)
wct_out = evaluator.filter_invalid_results(wct_out_raw, x_std=None)

wct_out.to_csv(base_output_path / "wct_out.csv")
logger.info(f"{len(wct_out)} valid samples after filtering and saved at {base_output_path / 'wct_out.csv'}")

wct_out_raw


2025-09-18 08:04:02.601 | INFO     | solhycool_modeling.evaluation:filter_invalid_results:14 - Filtering invalid results...
2025-09-18 08:04:02.604 | INFO     | solhycool_modeling.evaluation:filter_invalid_results:27 - Invalid rows detected: 6402 of 8614
2025-09-18 08:04:02.605 | INFO     | solhycool_modeling.evaluation:filter_invalid_results:29 - Remaining valid rows: 2212
2025-09-18 08:04:02.636 | INFO     | __main__:<module>:5 - 2212 valid samples after filtering and saved at ../results/model_inputs_sampling/andasol_90MW/wct_out.csv


,Tamb,HR,Twct_in,qwct,wwct,Twb,mw_ma_ratio,Tout,m_w_lost,evaluation_status,Ce
eval_index,,,,,,,,,,,
0,14.75,0.000971,27.0,1152.000000,32.103815,3.070432,0.500,12.258778,25137.208431,2,13.303067
1,14.75,0.000971,27.0,1152.000000,23.864511,3.070432,1.375,18.873751,13322.137777,2,7.077665
2,14.75,0.000971,27.0,1152.000000,22.151955,3.070432,2.250,17.131809,16348.591637,2,6.050759
3,14.75,0.000971,27.0,1152.000000,21.410259,3.070432,3.125,17.121929,16526.036316,2,5.633495
4,14.75,0.000971,27.0,1407.272727,35.218541,3.070432,0.500,12.258773,30707.394841,2,16.235340
...,...,...,...,...,...,...,...,...,...,...,...
8609,50.00,54.625129,43.0,3960.000000,92.136147,39.999675,0.500,NaN,NaN,2,NaN
8610,50.00,54.625129,43.0,3960.000000,35.628772,39.999675,1.375,NaN,NaN,2,NaN
8611,50.00,54.625129,43.0,3960.000000,28.935151,39.999675,2.250,NaN,NaN,2,NaN


In [8]:
var_ids = ["Tamb", "HR", "Twct_in", "qwct", "wwct", "Tout", "m_w_lost"]

fig = plot_samples(wct_out, Ncols=len(var_ids), var_ids=var_ids)
fig.update_layout(width=1000)

save_figure(fig, figure_path=base_output_path, figure_name="viz_evaluated_samples_wct", formats=["png", "html"], scale=2)
logger.info(f"Figure saved at {base_output_path}")



2025-09-18 08:04:27.147 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../results/model_inputs_sampling/andasol_90MW/viz_evaluated_samples_wct.png
2025-09-18 08:04:27.185 | INFO     | phd_visualizations:save_figure:41 - Figure saved in ../results/model_inputs_sampling/andasol_90MW/viz_evaluated_samples_wct.html
2025-09-18 08:04:27.186 | INFO     | __main__:<module>:7 - Figure saved at ../results/model_inputs_sampling/andasol_90MW


In [6]:
fig1, fig2 = evaluator.create_visualization(wct_out_raw=wct_out_raw, wct_out_filtered=wct_out)
fig1.show()
fig2.show()


2025-09-18 08:02:47.240 | INFO     | solhycool_modeling.evaluation:create_visualization:668 - Creating visualization...


2025-09-18 08:02:47.327 | INFO     | solhycool_modeling.evaluation:create_visualization:785 - Visualization saved to ../results/model_inputs_sampling/andasol_90MW/wct_evaluation_results.html
2025-09-18 08:02:47.453 | INFO     | solhycool_modeling.evaluation:_create_summary_plot:814 - 3D visualization saved to ../results/model_inputs_sampling/andasol_90MW/wct_3d_results.html


## Other checks

In [15]:
# Achievable mw/ma ratios
for qwct  in np.linspace(params["qwct"][0], params["qwct"][1], 5):
    mw_ma_ratios = []
    fan_speeds = np.linspace(20, 100, 10)
    for fan_speed in fan_speeds:
        ma_kgps = cc_model.fan_speed_to_air_mass_flow_rate_fit(fan_speed, "andasol")
        mw_kgps = qwct / 3.6 / ma_kgps
        mw_ma_ratios.append(mw_kgps)
    logger.info(f"qwct={qwct:.1f} m3/h: mw/ma ratio in [{min(mw_ma_ratios):.2f}, {max(mw_ma_ratios):.2f}] for fan speed in [{min(fan_speeds):.1f}, {max(fan_speeds):.1f}] %")


2025-09-10 21:12:21.992 | INFO     | __main__:<module>:9 - qwct=1107.0 m3/h: mw/ma ratio in [0.48, 0.56] for fan speed in [20.0, 100.0] %
2025-09-10 21:12:21.996 | INFO     | __main__:<module>:9 - qwct=1820.2 m3/h: mw/ma ratio in [0.79, 0.93] for fan speed in [20.0, 100.0] %
2025-09-10 21:12:22.001 | INFO     | __main__:<module>:9 - qwct=2533.5 m3/h: mw/ma ratio in [1.10, 1.29] for fan speed in [20.0, 100.0] %
2025-09-10 21:12:22.005 | INFO     | __main__:<module>:9 - qwct=3246.8 m3/h: mw/ma ratio in [1.41, 1.65] for fan speed in [20.0, 100.0] %
2025-09-10 21:12:22.009 | INFO     | __main__:<module>:9 - qwct=3960.0 m3/h: mw/ma ratio in [1.72, 2.02] for fan speed in [20.0, 100.0] %
2025-09-10 21:12:21.996 | INFO     | __main__:<module>:9 - qwct=1820.2 m3/h: mw/ma ratio in [0.79, 0.93] for fan speed in [20.0, 100.0] %
2025-09-10 21:12:22.001 | INFO     | __main__:<module>:9 - qwct=2533.5 m3/h: mw/ma ratio in [1.10, 1.29] for fan speed in [20.0, 100.0] %
2025-09-10 21:12:22.005 | INFO    

# Old

## DC

In [11]:
from SALib.sample import sobol

N: int = 256


In [ ]:
@dataclass
class ModelInputsRange:
    Tamb: tuple[float, float] = (0.1, 50.)
    Tdc_in: tuple[float, float] = (25., 45.)
    qdc: tuple[float, float] = (6., 24.)
    wdc: tuple[float, float] = (11., 99.18)
    
# 1. Define problem
output_path = base_output_path / "dc.csv"
mip = ModelInputsRange()

var_ids = list(asdict(mip).keys())

problem = {
    'num_vars': len(asdict(mip)),
    'names': var_ids,
    'bounds': list(asdict(mip).values()),
    'outputs': ['Tdc_out']
}

# 2. Generate samples
param_values = sobol.sample(problem, N=N, calc_second_order=True)

df = pd.DataFrame(param_values, columns=var_ids)
len0 = len(df)

# Filter out invalid points
# Tin > Tamb
df = df[df["Tdc_in"] < df["Tamb"]]

logger.info(f"After filtering, removed {len0-len(df)} rows, from {len0} to {len(df)}")
display(df)
df.to_csv(output_path)
logger.info(f"Samples saved at {output_path}")

fig = plot_samples(df, var_ids, Ncols=len(var_ids))
display(fig)
save_figure(fig, figure_path=base_output_path, figure_name="viz_samples_dc", formats=["png"])


2025-04-21 09:10:40.648 | INFO     | __main__:<module>:31 - After filtering, removed 1286 rows, from 2560 to 1274


,Tamb,Tdc_in,qdc,wdc
0,40.364071,15.911473,17.965287,26.752296
2,40.364071,9.434187,17.965287,26.752296
3,40.364071,15.911473,17.407316,26.752296
4,40.364071,15.911473,17.965287,52.662090
5,40.364071,9.434187,17.407316,52.662090
...,...,...,...,...
2552,40.500587,25.064700,12.182225,89.453014
2555,40.500587,25.064700,13.367585,38.546102
2557,39.025198,25.064700,12.182225,38.546102
2558,39.025198,25.064700,13.367585,89.453014


2025-04-21 09:10:40.673 | INFO     | __main__:<module>:34 - Samples saved at ../results/model_inputs_sampling/pilot_plant_200kW/dc.csv


2025-04-21 09:10:42.483 | INFO     | phd_visualizations:save_figure:42 - Figure saved in ../results/model_inputs_sampling/pilot_plant_200kW/viz_samples_dc.png


## WCT

In [ ]:
@dataclass
class ModelInputsRange:
    Tamb: tuple[float, float] = (0.1, 50.)
    HR: tuple[float, float] = (10.33, 89.25)
    Twct_in: tuple[float, float] = (25., 45.)
    qwct: tuple[float, float] = (6., 24.)
    wwct: tuple[float, float] = (21.0, 93.4161)

# 1. Define problem
output_path = base_output_path / "wct.csv"
mip = ModelInputsRange()
var_ids = list(asdict(mip).keys())

problem = {
    'num_vars': len(asdict(mip)),
    'names': var_ids,
    'bounds': list(asdict(mip).values()),
    'outputs': ['Twct_out']
}

# 2. Generate samples
param_values = sobol.sample(problem, N=N, calc_second_order=True)

df = pd.DataFrame(param_values, columns=var_ids)
len0 = len(df)

# Filter out invalid points
# Tin > Twb
df = df[ df["Twct_in"] > get_Twb(df["Tamb"], df["HR"]/100, np.full(len(df), 101325)) ]

logger.info(f"After filtering, removed {len0-len(df)} rows, from {len0} to {len(df)}")
display(df)
df.to_csv(output_path)
logger.info(f"Samples saved at {output_path}")

fig = plot_samples(df, var_ids, Ncols=len(var_ids))
display(fig)
save_figure(fig, figure_path=base_output_path, figure_name="viz_samples_wct", formats=["png"])
